# Train Xenocomm on the reduced melanoma PDX dataset

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import xenocomm as xc

def resolve_notebook_dir() -> Path:
    for name in ("__vsc_ipynb_file__", "__session__"):
        value = globals().get(name)
        if value:
            path = Path(value).expanduser()
            if path.suffix == ".ipynb":
                return path.resolve().parent
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    raise RuntimeError()

NOTEBOOK_DIR = resolve_notebook_dir()
DATA_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
MOUSE_H5AD = DATA_DIR / "adata_mouse.h5ad"
HUMAN_H5AD = DATA_DIR / "adata_human.h5ad"
MODEL_PATH = OUTPUT_DIR / "model.npz"
STAGED_PATH = OUTPUT_DIR / "model.staged.npz"

missing = [path for path in (MOUSE_H5AD, HUMAN_H5AD) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Prepared dataset is missing. Run scripts/prepare_reduced_dataset.py first. "
        f"Missing: {missing}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
adata_mouse = ad.read_h5ad(MOUSE_H5AD)
adata_human = ad.read_h5ad(HUMAN_H5AD)
adata_mouse, adata_human

In [ ]:
NUM_STEPS = 250
BATCH_SIZE = 1024
NUM_EPOCHS = 10
MIX_RATE = 1.0
CUTOFF = -10
N_POSTERIOR_SAMPLES = 100
RUN_TRAINING = True
TRAINING_VERBOSE = "print"

In [ ]:
if RUN_TRAINING:
    model = xc.XenocommModel(
        adata_mouse,
        adata_human,
        mix_rate=MIX_RATE,
        cutoff=CUTOFF,
    )
    model.build_model()
    model.train(
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        verbose=TRAINING_VERBOSE,
    )
    model.save(str(MODEL_PATH))
else:
    if not MODEL_PATH.exists():
        raise FileNotFoundError("Set RUN_TRAINING=True or provide outputs/model.npz")
    model = xc.XenocommModel.load(str(MODEL_PATH), adata_mouse, adata_human)

MODEL_PATH

In [ ]:
model.train(
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    verbose=TRAINING_VERBOSE,
)

In [ ]:
samples = {key: np.asarray(value) for key, value in model.sample(N_POSTERIOR_SAMPLES).items()}
var_dict = {key: np.asarray(value) for key, value in model.get_parameters().items()}

payload = {
    "ligands": np.array(list(model.ligands), dtype=object),
    "receptors": np.array(list(model.receptors), dtype=object),
    "targets": np.array(list(model.targets), dtype=object),
    "mean_ligand_np": np.asarray(model.mean_ligand_np),
    "ligand_receptor_matrix_np": np.asarray(model.ligand_receptor_matrix_np),
    "receptor_target_matrix_np": np.asarray(model.receptor_target_matrix_np),
}
for key, value in samples.items():
    payload[f"s_{key}"] = value
for key, value in var_dict.items():
    payload[f"v_{key}"] = value

np.savez(str(STAGED_PATH), **payload)
STAGED_PATH

In [ ]:
enrichment = xc.enrichment_df(model, samples, var_dict)
top_ligands = (
    enrichment.sort_values("enrichment", ascending=False)
    .loc[:, ["ligand", "enrichment", "prob_enriched", "human_fraction"]]
    .head(10)
)
print("Top enriched ligands")
display(top_ligands)

receptor_marginal = xc.receptor_marginal_df(model, samples, var_dict)
top_receptors = (
    receptor_marginal.sort_values("delta", ascending=False)
    .loc[:, ["receptor", "delta", "mouse"]]
    .head(10)
)
print("Top receptor marginal activations")
display(top_receptors)


In [ ]:
history = getattr(model, "training_history", [])
print(f"Saved model: {MODEL_PATH}")
print(f"Saved staged analysis artifact: {STAGED_PATH}")
print(f"Ligands: {len(model.ligands):,}; receptors: {len(model.receptors):,}; targets: {len(model.targets):,}")
if history:
    print(history[-1])